In [ ]:
pip install pandas numpy scikit-learn torch openpyxl

In [ ]:
import pandas as pd
import numpy as np

df = pd.read_excel("nba_last_7_days.xlsx")

print(df.head())
print(df.columns)

In [ ]:
# points scored
df["points"] = np.where(df["shotResult"] == "Made", df["shotValue"], 0)

# field goal attempt
df["fga"] = np.where(df["actionType"] == "Shot", 1, 0)

# field goal made
df["fgm"] = np.where(df["shotResult"] == "Made", 1, 0)

# three pointers made
df["three_pm"] = np.where(
    (df["shotResult"] == "Made") & (df["shotValue"] == 3), 1, 0
)

In [ ]:
player_games = df.groupby(["personId", "gameId"]).agg({
    "points": "sum",
    "fga": "sum",
    "fgm": "sum",
    "three_pm": "sum"
}).reset_index()

player_games = player_games.sort_values(["personId", "gameId"])

print(player_games.head())

In [ ]:
from sklearn.preprocessing import MinMaxScaler

features = ["points","fga","fgm","three_pm"]

scaler = MinMaxScaler()
player_games[features] = scaler.fit_transform(player_games[features])

In [ ]:
WINDOW = 3

X = []
y = []

for player in player_games["personId"].unique():

    pdata = player_games[player_games["personId"] == player]
    pdata = pdata[features].values

    if len(pdata) <= WINDOW:
        continue

    for i in range(len(pdata) - WINDOW):
        X.append(pdata[i:i+WINDOW])
        y.append(pdata[i+WINDOW])

X = np.array(X)
y = np.array(y)

print("X shape:", X.shape)
print("y shape:", y.shape)

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [ ]:
import torch

X_train = torch.tensor(X_train, dtype=torch.float32)
X_test = torch.tensor(X_test, dtype=torch.float32)

y_train = torch.tensor(y_train, dtype=torch.float32)
y_test = torch.tensor(y_test, dtype=torch.float32)

In [ ]:
import torch.nn as nn

class PlayerLSTM(nn.Module):

    def __init__(self, input_size, hidden_size, num_layers):
        super().__init__()

        self.lstm = nn.LSTM(
            input_size,
            hidden_size,
            num_layers,
            batch_first=True
        )

        self.fc = nn.Linear(hidden_size, input_size)

    def forward(self, x):

        out, _ = self.lstm(x)

        out = out[:, -1, :]

        out = self.fc(out)

        return out

In [ ]:
input_size = len(features)

model = PlayerLSTM(
    input_size=input_size,
    hidden_size=64,
    num_layers=2
)

criterion = nn.MSELoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001
)

In [ ]:
EPOCHS = 30

for epoch in range(EPOCHS):

    model.train()

    outputs = model(X_train)

    loss = criterion(outputs, y_train)

    optimizer.zero_grad()

    loss.backward()

    optimizer.step()

    if epoch % 5 == 0:
        print(f"Epoch {epoch} Loss {loss.item():.4f}")

In [ ]:
model.eval()

with torch.no_grad():

    preds = model(X_test)

    test_loss = criterion(preds, y_test)

print("Test Loss:", test_loss.item())

In [ ]:
sample = X_test[0].unsqueeze(0)

predicted = model(sample)

predicted = predicted.detach().numpy()

predicted = scaler.inverse_transform(predicted)

print(predicted)